In [1]:
import pandas as pd
import numpy as np
import joblib
import os

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load demand forecasting model
forecast_model = joblib.load(
    "../models/AquaSense_Demand_Forecasting_Model.pkl"
)

# Load forecasting feature names
forecast_features = joblib.load(
    "../models/AquaSense_Model_Features.pkl"
)

# Load anomaly detection model bundle
anomaly_bundle = joblib.load(
    "../models/AquaSense_Anomaly_Detection_Model_V3.pkl"
)

anomaly_model = anomaly_bundle["model"]
anomaly_threshold = anomaly_bundle["threshold"]
anomaly_features = anomaly_bundle["features"]

print("Both models loaded successfully!")

print("\nForecasting Features:", len(forecast_features))
print("Anomaly Detection Features:", len(anomaly_features))
print("Anomaly Threshold:", anomaly_threshold)

Both models loaded successfully!

Forecasting Features: 17
Anomaly Detection Features: 10
Anomaly Threshold: -0.14498929460587992


In [3]:
# Load feature-engineered dataset
integration_df = pd.read_csv(
    "../dataset/AquaSense_Feature_Engineered.csv"
)

integration_df["Timestamp"] = pd.to_datetime(
    integration_df["Timestamp"]
)

# Recreate chronological test split
split_index = int(len(integration_df) * 0.80)

test_data = integration_df.iloc[split_index:].copy()

print("Dataset loaded successfully!")
print("Test Records:", len(test_data))
print("Test Period:")
print(test_data["Timestamp"].min())
print("to")
print(test_data["Timestamp"].max())

Dataset loaded successfully!
Test Records: 9173
Test Period:
2020-03-15 19:00:00
to
2021-04-01 23:00:00


In [4]:
# Prepare forecasting inputs
X_forecast_test = test_data[forecast_features]

# Prepare anomaly detection inputs
X_anomaly_test_integration = test_data[anomaly_features]

# Demand predictions
forecast_predictions = forecast_model.predict(
    X_forecast_test
)

# Anomaly scores
anomaly_scores = anomaly_model.decision_function(
    X_anomaly_test_integration
)

# Classify anomalies using saved threshold
anomaly_predictions = np.where(
    anomaly_scores < anomaly_threshold,
    -1,
    1
)

print("Both models executed successfully!")

Both models executed successfully!


In [5]:
integration_results = test_data[
    ["Timestamp", "Consumption"]
].copy()

integration_results.rename(
    columns={"Consumption": "Actual_Consumption"},
    inplace=True
)

integration_results["Predicted_Consumption"] = (
    forecast_predictions
)

integration_results["Anomaly_Score"] = anomaly_scores

integration_results["Anomaly_Status"] = np.where(
    anomaly_predictions == -1,
    "Anomaly",
    "Normal"
)

display(integration_results.head(10))

,Timestamp,Actual_Consumption,Predicted_Consumption,Anomaly_Score,Anomaly_Status
36691,2020-03-15 19:00:00,2755,2623.64,0.076624,Normal
36692,2020-03-15 20:00:00,2474,2516.22,0.064666,Normal
36693,2020-03-15 21:00:00,1759,1834.66,0.044700,Normal
36694,2020-03-15 22:00:00,1515,1585.94,0.037703,Normal
36695,2020-03-15 23:00:00,1412,1491.86,0.026751,Normal
36696,2020-03-16 00:00:00,1509,1202.37,0.032008,Normal
36697,2020-03-16 01:00:00,1136,1020.71,0.037071,Normal
36698,2020-03-16 02:00:00,591,737.07,0.033015,Normal
36699,2020-03-16 03:00:00,735,558.15,0.031838,Normal
36700,2020-03-16 04:00:00,1210,836.28,0.044218,Normal


In [6]:
print("AquaSense AI Integration Summary")
print("--------------------------------")

print("Total Records:", len(integration_results))

print(
    "Forecasting MAE:",
    round(
        np.mean(
            np.abs(
                integration_results["Actual_Consumption"]
                - integration_results["Predicted_Consumption"]
            )
        ),
        2
    )
)

print(
    "Anomaly Alerts:",
    (integration_results["Anomaly_Status"] == "Anomaly").sum()
)

print(
    "Normal Records:",
    (integration_results["Anomaly_Status"] == "Normal").sum()
)

AquaSense AI Integration Summary
--------------------------------
Total Records: 9173
Forecasting MAE: 182.19
Anomaly Alerts: 263
Normal Records: 8910
